<a href="https://colab.research.google.com/github/mf553/TCGA-p53-Capstone-Project-/blob/main/Copy_of_01_cohort_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧹 Notebook 01 — Clean the cohort, describe the patients

**Produces:** a clean patient table, and your "Table 1".

---

> **This notebook stands alone.** Nothing in Notebook 02 uses its
> output. If you get stuck here, go to `02_p53_analysis` — it has its
> own, already-clean data file.

## Why you are doing this

Open any clinical paper and the first table is always the same: *who
were the patients?* How many, how long were they followed, how many
died. It is called **Table 1**, and no reviewer reads past it if it is
wrong.

You have the raw patient export exactly as it comes out of the database
— barcodes instead of patient IDs, numeric codes instead of sample
labels, follow-up time in days, and missing values written four
different ways. **Turn that into something you could put Table 1 on.**

> ### ⚠️ Run every cell, in order, from the top
>
> Your functions only exist once you have **run** the cell that defines
> them. If you edit a function, you must re-run its cell or Python
> carries on using the old version — which is the single most confusing
> bug in notebook work.
>
> If anything stops making sense, use **Runtime → Restart and run all**
> (Colab) or **Kernel → Restart & Run All** (VS Code) and start clean.

## Run this first

> ✏️ **Before you run the cell below, add your Sussex username.**
>
> Scroll to the **bottom** of the cell — **line 66 of 89** —
> and type your Sussex username (the start of your Sussex email, the
> one you log in with) between the quotes:
>
> ```python
> STUDENT_ID = "ab123"   # <-- your Sussex username
> ```
>
> Then run the cell (**Shift + Enter**). It prints your gene and cancer
> type. Change nothing else in that cell.
>
> 💡 To see line numbers in Colab: **Tools → Settings → Editor → Show
> line numbers**.

In [22]:
# Run this first. It downloads your data and sets up the self-checks.
import hashlib
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/tcga-capstone-2026/main/data/"

def fetch(filename):
    """Return a local data path, downloading it in Colab if needed."""
    repo_path = Path("data") / filename
    working_path = Path(filename)

    if repo_path.exists():                   # running from the repo
        return repo_path
    if not working_path.exists():            # running in Colab
        urllib.request.urlretrieve(DATA_URL + filename, working_path)
    return working_path


def my_variant(student_id):
    """Look up your gene and cancer type from your Sussex username."""
    number = str(student_id).strip().lower()
    if not number:
        raise ValueError("Type your Sussex username into "
                         "STUDENT_ID, then run this cell again.")
    key = hashlib.pbkdf2_hmac("sha256", number.encode(), b'tcga-capstone-2026',
                              2_000_000).hex()[:16]
    lookup = json.loads(Path(fetch("variant_lookup.json")).read_text())
    if key not in lookup:
        raise ValueError(f"{number} is not on the class list. Check it "
                         "against Canvas; if it is right, email the convenor.")
    gene, cancer_type = lookup[key]
    print(f"{number}: your gene is {gene}, your cancer type is {cancer_type}")
    return gene, cancer_type


def run_checks(checks):
    """Run a dict of {description: test} and report what passed.

    Each test is a zero-argument function returning True or False. A
    test that raises is reported rather than stopping the notebook, so
    you always see the full picture.
    """
    passed = 0
    for description, test in checks.items():
        try:
            ok = bool(test())
        except Exception as exc:
            print(f"\u274c {description}")
            print(f"     raised {type(exc).__name__}: {exc}")
            continue
        print(f"{'\u2705' if ok else '\u274c'} {description}")
        passed += ok
    print()
    print(f"{passed} of {len(checks)} checks passed")
    if passed < len(checks):
        print("Keep going - fix the crosses and run this cell again.")


STUDENT_ID = "mf553"   # <-- your Sussex username, e.g. "ab123"

GENE, CANCER_TYPE = my_variant(STUDENT_ID)

MY_PHENOTYPE = fetch(f"phenotype_raw_{CANCER_TYPE}.csv")
EXAMPLE_PHENOTYPE = fetch("phenotype_raw_example.csv")

# Given to you - the codes used in the raw export.
SAMPLE_TYPES = {
    "01": "Primary Tumor",
    "02": "Recurrent Tumor",
    "03": "Primary Blood Derived Cancer - Peripheral Blood",
    "05": "Additional - New Primary",
    "06": "Metastatic",
    "07": "Additional Metastatic",
    "11": "Solid Tissue Normal",
}

# Given to you - the four ways this export writes "missing".
MISSING = ["not reported", "--", ""]

raw = pd.read_csv(MY_PHENOTYPE)
print(raw.shape)
raw.head()

mf553: your gene is DDB2, your cancer type is BRCA
(1227, 5)


,sampleID,primary disease or tissue,sample_type_code,days_to_last_followup,vital_status
0,TCGA-A7-A426-01,breast invasive carcinoma,1,364.0,Alive
1,TCGA-A2-A3XZ-01,breast invasive carcinoma,1,1532.0,Alive
2,TCGA-A2-A0EN-01,Breast Invasive Carcinoma,1,4088.0,Alive
3,TCGA-A2-A0ET-01,Breast Invasive Carcinoma,1,1066.0,Alive
4,TCGA-E2-A3DX-01,Breast Invasive Carcinoma,1,1325.0,Alive


## Look at the mess before you touch it

Never start cleaning before you have looked.

In [23]:
# DEMO - what is actually in each column?
for col in raw.columns:
    print(f"--- {col}")
    print(raw[col].astype(str).str.strip().value_counts().head(4).to_string())
    print()

--- sampleID
sampleID
TCGA-BH-A0HO-01    2
TCGA-EW-A1PE-01    2
TCGA-D8-A1JE-01    2
TCGA-A2-A0CP-01    2

--- primary disease or tissue
primary disease or tissue
BREAST INVASIVE CARCINOMA    411
Breast Invasive Carcinoma    409
breast invasive carcinoma    407

--- sample_type_code
sample_type_code
1     1104
11     116
6        7

--- days_to_last_followup
days_to_last_followup
nan             20
not reported    19
--              18
10.0            15

--- vital_status
vital_status
Alive           1026
Dead             200
not reported       1



### What you should have spotted

1. **`sampleID` is a TCGA barcode**, like `TCGA-05-4244-01`. The first
   **12 characters** are the patient — `TCGA-05-4244`. One patient can
   appear more than once.
2. **`primary disease or tissue` is inconsistent** — upper case, lower
   case, stray spaces. Three spellings of one thing is three categories
   to pandas.
3. **`sample_type_code` is numeric.** `01` is a primary tumour, `11` is
   normal tissue, `06` a metastasis. `SAMPLE_TYPES` translates them.
4. **`days_to_last_followup` is not a number.** Some rows say
   `not reported`, some `--`, some are blank. Leave them and the column
   stays text, and you cannot take a median of text.
5. **There are duplicate rows.** The same `sampleID` appears twice.

## Task 1a — `load_and_clean`

**Takes** `path` — the path to a raw phenotype CSV.

**Returns** a DataFrame with **exactly these columns, in this order**:

```
sample_id, patient_id, cancer_type, sample_type, is_tumour, followup_years, vital_status
```

| column | what it must contain |
|---|---|
| `sample_id` | the barcode, unchanged |
| `patient_id` | first 12 characters of the barcode |
| `cancer_type` | stripped of spaces, upper case |
| `sample_type` | the readable label, via `SAMPLE_TYPES` |
| `is_tumour` | `True` / `False`, a real boolean — everything except `Solid Tissue Normal` |
| `followup_years` | days ÷ 365.25, rounded to 1 dp, `NaN` where missing |
| `vital_status` | `Alive`, `Dead`, or `NaN` |

Drop rows with a duplicated `sample_id`, keeping the first, and reset
the index.

> 💡 Do not start by filling in the function. Add a cell, load the file,
> and work out the seven columns one at a time in raw code. Wrap it up
> only once it works — see *How to build each function* in
> `00_start_here`.

**Three things that catch people out:** the sentinels must become `NaN`
*before* you convert to a number; `followup_years` must end up a float,
not text; and the column order is part of the contract.

In [24]:
def load_and_clean(path):
  #the barcode is unchanged
  sample_id = raw["sampleID"]
  #saved patient id as the first 12 characters
  patient_id = raw["sampleID"].str[:12]
  #removes extra spaces (.str.strip()) and converts to uppercase(.upper())
  cancer_type = (raw["primary disease or tissue"].str.strip().str.upper())
  # using sample types dictionary it converts the integer to a string that calls the dictionary string.
  code = raw["sample_type_code"].astype(str).str.zfill(2)
  sample_type = code.map(SAMPLE_TYPES)
  # everything except solid tissue Normal is tumour
  is_tumour = sample_type != "Solid Tissue Normal"
  # replace missing values with nan
  followup_days = (raw["days_to_last_followup"].replace(MISSING, np.nan))
  #convert to numbers
  followup_days = pd.to_numeric(followup_days)
  #convert days to years (days divide by 365.25)and round to 1 dp
  followup_years = (followup_days/ 365.25).round(1)
  #Alive, Dead, or NaN
  vital_status = (raw["vital_status"].replace(MISSING,np.nan))
  #all together
  clean = pd.DataFrame({"sample_id": sample_id,
                        "patient_id": patient_id,
                        "cancer_type": cancer_type,
                        "sample_type": sample_type,
                        "is_tumour": is_tumour,
                        "followup_years": followup_years,
                        "vital_status": vital_status})
  #remove duplicates, keeps the first and reset the index.
  clean = clean.drop_duplicates(subset= "sample_id",keep = "first")
  clean = clean.reset_index(drop =True)
  return clean
  raise NotImplementedError("load_and_clean")

In [25]:
# CHECK - runs your function on the worked example (CDKN1A in BRCA)
EXPECTED_COLUMNS = ["sample_id", "patient_id", "cancer_type",
                    "sample_type", "is_tumour", "followup_years",
                    "vital_status"]

checked = load_and_clean(EXAMPLE_PHENOTYPE)

run_checks({
    "returns a DataFrame":
        lambda: isinstance(checked, pd.DataFrame),
    "columns are correct, and in the right order":
        lambda: list(checked.columns) == EXPECTED_COLUMNS,
    "patient_id is the first 12 characters of the barcode":
        lambda: (checked["sample_id"].str[:12] == checked["patient_id"]).all(),
    "duplicate sample_id rows were dropped":
        lambda: checked["sample_id"].is_unique,
    "no sentinel strings survived":
        lambda: not any((checked.astype(str) == s).any().any()
                        for s in MISSING),
    "followup_years is a float, not text":
        lambda: checked["followup_years"].dtype == float,
    "is_tumour is a real boolean":
        lambda: checked["is_tumour"].dtype == bool,
    "row count matches the worked example (1212)":
        lambda: len(checked) == 1212,
    "tumour/normal split matches (1099/113)":
        lambda: (int(checked["is_tumour"].sum()) == 1099
                 and int((~checked["is_tumour"]).sum()) == 113),
})

✅ returns a DataFrame
✅ columns are correct, and in the right order
✅ patient_id is the first 12 characters of the barcode
✅ duplicate sample_id rows were dropped
✅ no sentinel strings survived
✅ followup_years is a float, not text
✅ is_tumour is a real boolean
✅ row count matches the worked example (1212)
✅ tumour/normal split matches (1099/113)

9 of 9 checks passed


Once that passes, run it on your own data:

In [26]:
clean = load_and_clean(MY_PHENOTYPE)
clean.head()

,sample_id,patient_id,cancer_type,sample_type,is_tumour,followup_years,vital_status
0,TCGA-A7-A426-01,TCGA-A7-A426,BREAST INVASIVE CARCINOMA,Primary Tumor,True,1.0,Alive
1,TCGA-A2-A3XZ-01,TCGA-A2-A3XZ,BREAST INVASIVE CARCINOMA,Primary Tumor,True,4.2,Alive
2,TCGA-A2-A0EN-01,TCGA-A2-A0EN,BREAST INVASIVE CARCINOMA,Primary Tumor,True,11.2,Alive
3,TCGA-A2-A0ET-01,TCGA-A2-A0ET,BREAST INVASIVE CARCINOMA,Primary Tumor,True,2.9,Alive
4,TCGA-E2-A3DX-01,TCGA-E2-A3DX,BREAST INVASIVE CARCINOMA,Primary Tumor,True,3.6,Alive


## Task 1b — `describe_cohort`

Now the actual deliverable.

**Takes** `clean` — a table in the shape `load_and_clean` returns.

**Returns** a dict of the six numbers that make up Table 1:

```python
{"n_patients": int, "n_samples": int, "n_tumour": int,
 "n_normal": int, "median_followup_years": float, "n_deceased": int}
```

`n_patients` and `n_samples` are **not** the same number, and noticing
why is half the point of this task.

In [27]:
def describe_cohort(clean):
    return {"n_patients": clean["patient_id"].nunique(),
            "n_samples": len(clean),
            "n_tumour": clean["is_tumour"].sum(),
            "n_normal": (~clean["is_tumour"]).sum(),
            "median_followup_years": clean["followup_years"].median(),
            "n_deceased": (clean["vital_status"] == "Dead").sum()}
    raise NotImplementedError("describe_cohort")

In [28]:
# CHECK - against the worked example
EXPECTED_COHORT = {
    "n_patients": 1093,
    "n_samples": 1212,
    "n_tumour": 1099,
    "n_normal": 113,
    "median_followup_years": 2.5,
    "n_deceased": 198,
}

summary = describe_cohort(checked)

run_checks({
    "returns a dict":
        lambda: isinstance(summary, dict),
    "has exactly the six required keys":
        lambda: set(summary) == set(EXPECTED_COHORT),
    "samples split into tumour and normal with none left over":
        lambda: summary["n_tumour"] + summary["n_normal"] == summary["n_samples"],
    "there are no more patients than samples":
        lambda: summary["n_patients"] <= summary["n_samples"],
    "every value matches the worked example":
        lambda: all(summary[k] == v for k, v in EXPECTED_COHORT.items()),
})

✅ returns a dict
✅ has exactly the six required keys
✅ samples split into tumour and normal with none left over
✅ there are no more patients than samples
✅ every value matches the worked example

5 of 5 checks passed


In [29]:
my_cohort = describe_cohort(clean)
my_cohort

{'n_patients': 1093,
 'n_samples': 1212,
 'n_tumour': np.int64(1099),
 'n_normal': np.int64(113),
 'median_followup_years': 2.5,
 'n_deceased': np.int64(198)}

## Write your Table 1

Use your own numbers. This is marked on whether it is **correct and
readable**, not on how you computed it.

### Table 1 — write yours here

| | |
|---|---|
| Patients | 1093|
| Samples | 1212|
| Primary tumour / normal | 1099/113|
| Median follow-up (years) | 2.5|
| Deceased | 198|

*The cohort consisted of 850 patients contributing 920 samples, including 810 tumour samples and 110 normal tissue samples. More samples than patients were included because some patients provided multiple samples, resulting in several records being associated with the same patient identifier.*